# 03 · Baselines: mediana, TF-IDF + Ridge, embeddings + Ridge (+ regla de fechas)

**Etapa 4.** Ejecutar en un **entorno nuevo** de Colab con GPU T4 (*Entorno de ejecución → Desconectar y
borrar entorno*), no en el del notebook 01: así PyTorch no choca con PaddlePaddle.

Reglas: todos los modelos usan el split de `data/splits/`; los hiperparámetros se eligen **solo con
validación**; el test se predice y se guarda, pero se analiza recién en el notebook 05.

In [ ]:
!pip install -q "sentence-transformers==6.0.1" "transformers==5.16.1" "scikit-learn==1.9.0" "joblib>=1.4"

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/leruizdelavegat-max/HW_04_202602.git"
RAMA = "feature/pipeline"          # cambiar a "main" cuando se haga el merge del PR
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    RAIZ = Path("/content/drive/MyDrive/HW_04_202602")
    if not (RAIZ / ".git").exists():
        subprocess.run(["git", "clone", "-b", RAMA, REPO_URL, str(RAIZ)], check=True)
    else:
        subprocess.run(["git", "-C", str(RAIZ), "fetch", "origin", RAMA], check=False)
        subprocess.run(["git", "-C", str(RAIZ), "checkout", RAMA], check=False)
        subprocess.run(["git", "-C", str(RAIZ), "pull", "--ff-only", "origin", RAMA], check=False)
else:
    RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))

In [ ]:
import json
import time

import joblib
import numpy as np
import pandas as pd
try:
    import torch
except ImportError as e:
    raise RuntimeError("PyTorch no carga: este entorno tiene restos del notebook 01 (PaddlePaddle). Usa "
                       "«Entorno de ejecución → Desconectar y borrar entorno de ejecución» y vuelve a ejecutar.") from e
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

from src import config, modelos, reglas

df = pd.read_parquet(config.PROCESADOS / "dataset.parquet")
run = json.loads((config.PROCESADOS / "run_config.json").read_text())
TOPE = run["tope"]
for nombre in ("train", "val", "test"):          # el split se lee de disco, igual para todos
    ids = set(pd.read_csv(config.SPLITS / f"{nombre}_ids.csv", dtype=str)["id"])
    assert (df.loc[df.split == nombre, "id"].isin(ids)).all()
tr, va, te = (df[df.split == s].reset_index(drop=True) for s in ("train", "val", "test"))
print(len(tr), len(va), len(te), "· tope", TOPE, "· GPU:", torch.cuda.is_available())

PRED = config.OUTPUT / "predicciones"
PRED.mkdir(exist_ok=True)
tiempos = {}
recorte = lambda p: np.clip(p, 0, TOPE)


def guardar(nombre, p_va, p_te, p_te_ref=None):
    out = pd.concat([pd.DataFrame({"id": va.id, "split": "val", "y": va.y, "pred": p_va}),
                     pd.DataFrame({"id": te.id, "split": "test", "y": te.y, "pred": p_te,
                                   "pred_ref": p_te_ref if p_te_ref is not None else np.nan})])
    out.to_csv(PRED / f"{nombre}.csv", index=False)
    print(f"{nombre:<14} MAE val = {mean_absolute_error(va.y, p_va):.3f}")

## Baseline 0 · Mediana del train

El piso: cualquier modelo que no le gane no aprendió nada útil.

In [ ]:
t0 = time.perf_counter()
med = modelos.Mediana().fit(tr.texto_final, tr.y)
tiempos["mediana"] = (time.perf_counter() - t0, 0.0)
guardar("mediana", med.predict(va), med.predict(te), med.predict(te))
(config.MODELOS / "mediana.json").write_text(json.dumps({"valor": med.valor_}))

## Baseline 1 · TF-IDF + Ridge

**¿Números y años, sí o no?** Sí por defecto: un token «2009» o «present» dice cuándo empezó la carrera, y
eso es justamente la señal del target. La variante sin dígitos (`digitos=False`, cada dígito → `#`) se
evalúa en la Pregunta 3 para medir cuánto depende el modelo de **ver** los años.

Búsqueda en validación: `alpha` × (solo palabras 1–2 | palabras + caracteres 3–5).

In [ ]:
grid = []
for chars in (False, True):
    for alpha in (0.3, 1.0, 3.0, 10.0):
        m = modelos.tfidf_ridge(alpha=alpha, chars=chars).fit(tr.texto_final, tr.y)
        grid.append({"chars": chars, "alpha": alpha, "MAE_val": mean_absolute_error(va.y, recorte(m.predict(va.texto_final)))})
grid = pd.DataFrame(grid).sort_values("MAE_val")
grid.to_csv(config.TABLAS / "grid_tfidf.csv", index=False)
display(grid.head(8))
mejor_tfidf = grid.iloc[0].to_dict()

t0 = time.perf_counter()
tfidf = modelos.tfidf_ridge(alpha=mejor_tfidf["alpha"], chars=bool(mejor_tfidf["chars"])).fit(tr.texto_final, tr.y)
t_fit = time.perf_counter() - t0
t0 = time.perf_counter(); p_te = recorte(tfidf.predict(te.texto_final)); t_inf = (time.perf_counter() - t0) / len(te)
tiempos["tfidf_ridge"] = (t_fit, t_inf)
guardar("tfidf_ridge", recorte(tfidf.predict(va.texto_final)), p_te, recorte(tfidf.predict(te.texto_ref)))
joblib.dump(tfidf, config.MODELOS / "tfidf_ridge.joblib")

In [ ]:
# ¿Qué n-gramas empujan la predicción? (solo la parte de palabras)
vec = tfidf.named_steps["tfidf"].transformer_list[0][1]
coef = tfidf.named_steps["ridge"].coef_[: len(vec.vocabulary_)]
voc = np.array(vec.get_feature_names_out())
orden = np.argsort(coef)
display(pd.DataFrame({"suben_experiencia": voc[orden[-20:]][::-1], "coef+": coef[orden[-20:]][::-1].round(2),
                      "bajan_experiencia": voc[orden[:20]], "coef-": coef[orden[:20]].round(2)}))

## Baseline 2 · Embeddings congelados + Ridge

Encoder: `sentence-transformers/all-MiniLM-L6-v2` (384 dimensiones, sin fine-tuning). Su límite es 256
tokens y un CV tiene varios cientos, así que **se trocea** en ventanas de ~180 palabras con solape de 30 y
se **promedian** los embeddings de los trozos: ninguna parte del CV se pierde por truncamiento.

In [ ]:
from sentence_transformers import SentenceTransformer
ENCODER = "sentence-transformers/all-MiniLM-L6-v2"
enc = SentenceTransformer(ENCODER, device="cuda" if torch.cuda.is_available() else "cpu")

t0 = time.perf_counter()
E_tr = modelos.embeber(list(tr.texto_final), enc)
t_emb_tr = time.perf_counter() - t0
E_va = modelos.embeber(list(va.texto_final), enc)
t0 = time.perf_counter(); E_te = modelos.embeber(list(te.texto_final), enc); t_emb_te = time.perf_counter() - t0
E_te_ref = modelos.embeber(list(te.texto_ref), enc)

grid_e = pd.DataFrame([{"alpha": a, "MAE_val": mean_absolute_error(va.y, recorte(Ridge(alpha=a).fit(E_tr, tr.y).predict(E_va)))}
                       for a in (0.01, 0.1, 0.3, 1.0, 3.0, 10.0)]).sort_values("MAE_val")
grid_e.to_csv(config.TABLAS / "grid_embeddings.csv", index=False)
display(grid_e)
t0 = time.perf_counter()
ridge_e = Ridge(alpha=float(grid_e.iloc[0].alpha)).fit(E_tr, tr.y)
tiempos["embeddings_ridge"] = (t_emb_tr + time.perf_counter() - t0, t_emb_te / len(te))
guardar("embeddings_ridge", recorte(ridge_e.predict(E_va)), recorte(ridge_e.predict(E_te)), recorte(ridge_e.predict(E_te_ref)))
joblib.dump({"encoder": ENCODER, "ridge": ridge_e}, config.MODELOS / "embeddings_ridge.joblib")

## Bonus · Regla de conteo de fechas

Sin aprendizaje: busca rangos «Mar 2014 – Jun 2018», «2015 - Present», «03/2012 - 2016», descarta los que
están cerca de palabras de educación, **fusiona solapamientos** y suma. El único parámetro es el año que
significa «Present» en estos CVs sintéticos; se calibra con el train.

In [ ]:
cal = {a: mean_absolute_error(tr.y, [reglas.estimar(t, a, TOPE) for t in tr.texto_final]) for a in range(2015, 2027)}
ANIO_ACTUAL = min(cal, key=cal.get)
print("Año «Present» calibrado:", ANIO_ACTUAL, "· MAE train:", round(cal[ANIO_ACTUAL], 3))
regla = lambda s: np.array([reglas.estimar(t, ANIO_ACTUAL, TOPE) for t in s])
t0 = time.perf_counter(); p_te = regla(te.texto_final); t_inf = (time.perf_counter() - t0) / len(te)
tiempos["regla_fechas"] = (0.0, t_inf)
guardar("regla_fechas", regla(va.texto_final), p_te, regla(te.texto_ref))
(config.MODELOS / "regla.json").write_text(json.dumps({"anio_actual": ANIO_ACTUAL}))
print("CVs donde la regla no encontró ningún rango (test):", int((p_te == 0).sum()))

## Pregunta 3 · Sensibilidad (baselines)

Se reentrena cada baseline cambiando **una** decisión a la vez y se compara el MAE de test:

- **Dígitos en TF-IDF:** con vs sin.
- **Limpieza del OCR:** `crudo` vs `limpio` vs `final` (sin líneas de baja confianza).

In [ ]:
sens = []
for variante in ("texto_crudo", "texto_limpio", "texto_final"):
    m = modelos.tfidf_ridge(alpha=mejor_tfidf["alpha"], chars=bool(mejor_tfidf["chars"])).fit(tr[variante], tr.y)
    sens.append({"modelo": "tfidf_ridge", "decision": "limpieza", "variante": variante,
                 "MAE_test": mean_absolute_error(te.y, recorte(m.predict(te[variante])))})
    Etr = modelos.embeber(list(tr[variante]), enc); Ete = modelos.embeber(list(te[variante]), enc)
    r = Ridge(alpha=float(grid_e.iloc[0].alpha)).fit(Etr, tr.y)
    sens.append({"modelo": "embeddings_ridge", "decision": "limpieza", "variante": variante,
                 "MAE_test": mean_absolute_error(te.y, recorte(r.predict(Ete)))})
    sens.append({"modelo": "regla_fechas", "decision": "limpieza", "variante": variante,
                 "MAE_test": mean_absolute_error(te.y, [reglas.estimar(t, ANIO_ACTUAL, TOPE) for t in te[variante]])})
for digitos in (True, False):
    m = modelos.tfidf_ridge(alpha=mejor_tfidf["alpha"], chars=bool(mejor_tfidf["chars"]), digitos=digitos).fit(tr.texto_final, tr.y)
    sens.append({"modelo": "tfidf_ridge", "decision": "digitos", "variante": "con dígitos" if digitos else "sin dígitos (#)",
                 "MAE_test": mean_absolute_error(te.y, recorte(m.predict(te.texto_final)))})
sens = pd.DataFrame(sens)
sens.to_csv(config.TABLAS / "sensibilidad_baselines.csv", index=False)
sens.pivot_table(index=["decision", "variante"], columns="modelo", values="MAE_test").round(3)

In [ ]:
(config.TABLAS / "tiempos_baselines.json").write_text(json.dumps(tiempos, indent=2))
(config.TABLAS / "hiperparametros_baselines.json").write_text(json.dumps({
    "tfidf": {**mejor_tfidf, "ngram_palabras": [1, 2], "ngram_caracteres": [3, 5] if mejor_tfidf["chars"] else None,
              "digitos": True, "sublinear_tf": True, "min_df": 2},
    "embeddings": {"encoder": ENCODER, "alpha": float(grid_e.iloc[0].alpha), "trozo_palabras": 180, "solape": 30,
                   "agregacion": "promedio"},
    "regla": {"anio_actual": ANIO_ACTUAL}}, indent=2, default=float))
print(json.dumps(tiempos, indent=2))

In [ ]:
import shutil
if EN_COLAB:
    shutil.make_archive("/content/entregables_03", "zip", RAIZ, "output")
    from google.colab import files
    files.download("/content/entregables_03.zip")